# Timing CPU and GPU inference

Load datasets for testing performance in inference

In [1]:
model_path = "Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras"

x_test_path = 'Data/x_test.npy'
y_test_path = 'Data/y_test.npy'

iterations = 30

In [2]:
# Check if GPU is available
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if not device_name:
  compute = 'CPU'
  timings_path = 'Timings/CPU/'
  print('GPU device not found')
else: 
  compute = 'GPU'
  timings_path = 'Timings/GPU/'
  print('Found GPU at: {}'.format(device_name))

2026-06-09 11:43:43.768654: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-06-09 11:43:43.768654: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU device not found


In [3]:
import os
import sys
import time
import numpy as np
# Load input from .npy file
x_test = np.load(x_test_path)
y_test = np.load(y_test_path)

In [19]:
def test_dut():
    start = time.perf_counter()
    y_dut = model.predict(x_test, verbose=0)
    end = time.perf_counter() - start

    return [y_dut, end]

In [20]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

Run the actual inference

In [22]:
from keras.models import load_model
import hgq.layers
from hgq.utils import trace_minmax

model = load_model(model_path)
# Calibrate datalane in HGQ2-model since it has layers with WRAP
trace_minmax(model, x_test, verbose=True)


dense_0: 455
dense_1: 194
dense_2: 191
dense_3: 161
Total: 1001


1001

In [23]:

timestamp = time.strftime("%Y%m%d_%H%M%S")
nr_samples = x_test.shape[0]
timings = []
for i in range(iterations):    
    # do inference
    result = test_dut()
    y_dut = result[0] 
    timings.append(result[1])

    acc = cal_accuracy(y_dut)
    print(f"\nTime: {result[1]}, Acc: {acc}")

2026-06-09 11:32:59.632887: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 4.131686539010843, Acc: 0.7435530120481928

Time: 3.1377803290088195, Acc: 0.7435530120481928

Time: 3.0995962909946684, Acc: 0.7435530120481928

Time: 3.071753569005523, Acc: 0.7435530120481928

Time: 3.122997161000967, Acc: 0.7435530120481928

Time: 3.1236460590153, Acc: 0.7435530120481928

Time: 3.0713590630039107, Acc: 0.7435530120481928

Time: 3.0585931170207914, Acc: 0.7435530120481928

Time: 3.1341066710010637, Acc: 0.7435530120481928

Time: 3.0437372800079174, Acc: 0.7435530120481928

Time: 3.032290770992404, Acc: 0.7435530120481928

Time: 3.0829323859943543, Acc: 0.7435530120481928

Time: 3.1192601610091515, Acc: 0.7435530120481928

Time: 3.065685235982528, Acc: 0.7435530120481928

Time: 3.04261283899541, Acc: 0.7435530120481928

Time: 3.068858550977893, Acc: 0.7435530120481928


2026-06-09 11:33:49.739687: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Time: 3.0570729649916757, Acc: 0.7435530120481928

Time: 3.1015986779821105, Acc: 0.7435530120481928

Time: 3.067245649988763, Acc: 0.7435530120481928

Time: 3.065915720013436, Acc: 0.7435530120481928

Time: 3.1162491769937333, Acc: 0.7435530120481928

Time: 3.0905539439991117, Acc: 0.7435530120481928

Time: 3.098293173010461, Acc: 0.7435530120481928

Time: 3.0817534049856476, Acc: 0.7435530120481928

Time: 3.112539353023749, Acc: 0.7435530120481928

Time: 3.1181592070206534, Acc: 0.7435530120481928

Time: 3.1122982910019346, Acc: 0.7435530120481928

Time: 3.0837836859864183, Acc: 0.7435530120481928

Time: 3.1073696420062333, Acc: 0.7435530120481928

Time: 3.094484608998755, Acc: 0.7435530120481928


FileNotFoundError: [Errno 2] No such file or directory: 'Timings/CPU/timings_dataset-830000_30iterations_20260609_113256.txt'

In [25]:
timing_results_path = f"{timings_path}timings_dataset-{nr_samples}_{iterations}iterations_{timestamp}.txt"
np.savetxt(timing_results_path,timings)

In [26]:
# Compute statistics for collected timings (convert to ms)
import json
import numpy as np

arr = np.array(timings)
arr_ms = arr * 1e3

stats = {
    'count': int(arr_ms.size),
    'mean_ms': float(np.mean(arr_ms)),
    'median_ms': float(np.median(arr_ms)),
    'std_ms': float(np.std(arr_ms, ddof=0)),
    'min_ms': float(np.min(arr_ms)),
    'max_ms': float(np.max(arr_ms)),
    'p5_ms': float(np.percentile(arr_ms, 5)),
    'p95_ms': float(np.percentile(arr_ms, 95)),
    'inference-rate (MHz)': float((nr_samples * 1000 / np.median(arr_ms) / 1000000)),
}

# Print summary
print('Timing statistics (ms):')
for k,v in stats.items():
    print(f"{k}: {v}")

# Save JSON summary next to timings file if available
try:
    base = timing_results_path
    out = base.rsplit('.',1)[0] + '_stats.json'
except NameError:
    out = 'timing_stats.json'

with open(out, 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Saved timing summary to: {out}')


Timing statistics (ms):
count: 30
mean_ms: 3123.8071174341408
median_ms: 3092.5192764989333
std_ms: 189.20826507637886
min_ms: 3032.290770992404
max_ms: 4131.686539010843
p5_ms: 3043.1188374510384
p95_ms: 3136.1271829053294
inference-rate (MHz): 0.2683895962451849
Saved timing summary to: Timings/CPU/timings_dataset-830000_30iterations_20260609_113256_stats.json


[Tensorflow profiler](https://github.com/tensorflow/tensorboard/blob/master/docs/tensorboard_profiling_keras.ipynb) 

In [ ]:
#!pip install -U tensorboard_plugin_profile

In [27]:
import tensorflow as tf
tf.profiler.experimental.start("logs/profile")

model.predict(x_test, batch_size=256)

tf.profiler.experimental.stop()


2026-06-09 11:36:14.375320: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:103] Profiler session initializing.
2026-06-09 11:36:14.375341: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:118] Profiler session started.


3243/3243 ━━━━━━━━━━━━━━━━━━━━ 1s 389us/step


2026-06-09 11:36:15.696351: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:68] Profiler session collecting data.
2026-06-09 11:36:16.363972: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:136] Profiler session tear down.
2026-06-09 11:36:16.366006: I external/local_xla/xla/tsl/profiler/rpc/client/save_profile.cc:150] Collecting XSpace to repository: logs/profile/plugins/profile/2026_06_09_11_36_16/KrissDEV.xplane.pb


In [28]:

# Load the TensorBoard notebook extension.
%load_ext tensorboard
# Launch TensorBoard and navigate to the Profile tab to view performance profile
%tensorboard --logdir=logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 287237), started 0:00:39 ago. (Use '!kill 287237' to kill it.)